# Read sas files in spark 4

Before spark 4, we use this https://github.com/saurfang/spark-sas7bdat repo to read sas file.

But the maintainer of the repo abandoned the project.

As a result, we move to a new repo.

https://github.com/Making-Sense-Info/spark-sas7bdat.git

To build the jar:

1. git clone https://github.com/Making-Sense-Info/spark-sas7bdat.git
2. sbt -Dspark.version=4.1.2 ++2.13.17 assembly (for linux)
3. $env:SBT_OPTS="-Dspark.version=4.1.2"; sbt ++2.13.17 assembly (for windows powershell)

After sbt build, you should have a `fat jar` spark-sas7bdat-4.0.0-s_2.13-assembly.jar, which can run in air-gapped server.


In [10]:
from pyspark.sql import SparkSession

In [11]:
import os
import pyspark

print("PySpark:", pyspark.__version__)
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))
print("PATH:")
print(os.environ.get("PATH"))

PySpark: 4.1.3
JAVA_HOME: C:\Users\pliu\AppData\Local\installed-spark\jdk-17.0.18
SPARK_HOME: C:\Users\pliu\AppData\Local\installed-spark\spark-4.1.3
HADOOP_HOME: C:\Users\pliu\AppData\Local\installed-spark\hadoop-3.4.3
PATH:
C:\Users\pliu\Documents\git\PySparkTutorial\.venv\Scripts;C:\Program Files (x86)\VMware\VMware Workstation\bin\;C:\WINDOWS\system32;C:\WINDOWS;C:\WINDOWS\System32\Wbem;C:\WINDOWS\System32\WindowsPowerShell\v1.0\;C:\WINDOWS\System32\OpenSSH\;C:\Program Files\dotnet\;C:\Program Files (x86)\NVIDIA Corporation\PhysX\Common;C:\Users\pliu\AppData\Local\installed-spark\jdk-17.0.18\bin;C:\Users\pliu\AppData\Local\installed-spark\hadoop-3.4.3\bin;C:\Users\pliu\AppData\Local\installed-spark\spark-4.1.3\bin;C:\Users\pliu\AppData\Local\Microsoft\WindowsApps;"C:\WINDOWS\system32;C:\WINDOWS;C:\WINDOWS\System32\Wbem;C:\WINDOWS\System32\WindowsPowerShell\v1.0;C:\WINDOWS\System32\OpenSSH:C:\Program Files\dotnet;C:\Program Files (x86)\NVIDIA Corporation\PhysX\Common;";C:\Users\pliu

In [3]:
# in air gap situation, we need to load jar file manually.

# in below example, we load a thin read_sas jar, as it requires other dependencies, we need to
# load all jars
from pathlib import Path

jar_folder = Path("C:/Users/pliu/Documents/git/PySparkTutorial/jars/sas")
jar_list = [str(jar) for jar in jar_folder.iterdir() if jar.is_file()]
jar_path = ",".join(jar_list)

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Read SAS")
    .config("spark.jars", jar_path)
    .getOrCreate()
)

In [12]:
#
fat_jar_path = Path("C:/Users/pliu/Documents/git/PySparkTutorial/jars/spark-sas7bdat_2.13-4.0.1-assembly.jar")
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Read SAS")
    .config("spark.jars", fat_jar_path)
    .getOrCreate()
)

In [4]:
# with internet, we can delegate the jar loading to maven
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Read SAS")
    .config("spark.jars.packages", "info.making-sense:spark-sas7bdat_2.13:4.0.0")
    .getOrCreate()
)

In [13]:
data_path = "C:/Users/pliu/Documents/data_set/sas_sample/dates.sas7bdat"
df = spark.read.format("info.makingsense.sas.spark").load(data_path)

In [14]:
df.printSchema()

df.show(50, truncate=False)

root
 |-- dt: timestamp (nullable = true)
 |-- string_dt: string (nullable = true)
 |-- timezone: string (nullable = true)
 |-- dates: date (nullable = true)
 |-- string_dates: string (nullable = true)
 |-- times: double (nullable = true)
 |-- string_times: string (nullable = true)
 |-- seconds: double (nullable = true)
 |-- missings: double (nullable = true)

+-------------------+-------------------+--------+----------+------------+-------+------------+-------------+--------+
|dt                 |string_dt          |timezone|dates     |string_dates|times  |string_times|seconds      |missings|
+-------------------+-------------------+--------+----------+------------+-------+------------+-------------+--------+
|1959-12-31 00:59:59|1959-12-30 23:59:59|UTC     |1959-12-30|1959-12-30  |86399.0|23:59:59    |-86401.0     |1.0     |
|1959-12-31 01:00:00|1959-12-31 00:00:00|UTC     |1959-12-31|1959-12-31  |0.0    |00:00:00    |-86400.0     |NULL    |
|1959-12-31 01:00:01|1959-12-31 00:00:01|U

In [16]:
# write data to local file system
out_path = "C:/Users/pliu/Documents/git/PySparkTutorial/data/tmp/dates"

df.write.mode("overwrite").parquet(out_path)

In [17]:
df2 = spark.read.parquet(out_path)
df2.show()

+-------------------+-------------------+--------+----------+------------+-------+------------+-------------+--------+
|                 dt|          string_dt|timezone|     dates|string_dates|  times|string_times|      seconds|missings|
+-------------------+-------------------+--------+----------+------------+-------+------------+-------------+--------+
|1959-12-31 00:59:59|1959-12-30 23:59:59|     UTC|1959-12-30|  1959-12-30|86399.0|    23:59:59|     -86401.0|     1.0|
|1959-12-31 01:00:00|1959-12-31 00:00:00|     UTC|1959-12-31|  1959-12-31|    0.0|    00:00:00|     -86400.0|    NULL|
|1959-12-31 01:00:01|1959-12-31 00:00:01|     UTC|1959-12-31|  1959-12-31|    1.0|    00:00:01|     -86399.0|     3.0|
|1960-01-01 00:59:59|1959-12-31 23:59:59|     UTC|1959-12-31|  1959-12-31|86399.0|    23:59:59|         -1.0|     4.0|
|1960-01-01 01:00:00|1960-01-01 00:00:00|     UTC|1960-01-01|  1960-01-01|    0.0|    00:00:00|          0.0|    NULL|
|1960-01-01 01:00:01|1960-01-01 00:00:01|     UT

In [9]:
spark.stop()